# Lakeview Dashboard Migration - Part 1: Setup & Configuration

## Overview

This is the first of three notebooks for migrating Databricks Lakeview dashboards between workspaces.

### Three-Notebook Workflow

1. **This Notebook (Setup)**: Configure authentication, create volume, prepare CSV mappings
2. **Notebook 2 (Export & Transform)**: Export dashboards and apply catalog/schema transformations
3. **Notebook 3 (Import & Migrate)**: Import to target workspace and restore permissions

## What This Notebook Does

- ✅ Install required libraries
- ✅ Configure authentication (OAuth RECOMMENDED, Service Principal, or PAT)
- ✅ Set up Databricks volume structure
- ✅ Create/validate CSV mapping file
- ✅ Test workspace connectivity
- ✅ Validate configuration

## Prerequisites

- Access to source and target Databricks workspaces
- Permissions to create volumes in Unity Catalog
- One of: Azure CLI (`az login`), Service Principal credentials, or PAT tokens

---

## Cell 1: Install Required Libraries

**Purpose:** Install the Databricks SDK and other required packages.

In [ ]:
# Install required packages
%pip install databricks-sdk pandas --quiet

# Restart Python kernel to load new packages
dbutils.library.restartPython()

## Cell 2: Import Libraries

**Purpose:** Import all required libraries and verify imports.

In [ ]:
import json
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from pathlib import Path
import pandas as pd

try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat, ImportFormat
    print("✅ All libraries imported successfully")
    print(f"   Databricks SDK version: Available")
    print(f"   Pandas version: {pd.__version__}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("   Please run Cell 1 to install required packages")

## Cell 3: Authentication Configuration

**Purpose:** Configure authentication method for both source and target workspaces.

### Choose Your Authentication Method

| Method | Setup | Best For | Recommended? |
|--------|-------|----------|-------------|
| **OAuth** | `az login` | Interactive use, most scenarios | **YES** |
| Service Principal | Store in secrets | Production, CI/CD, automation | For automation |
| PAT Token | Store in secrets | Quick tests, legacy | No |

### Instructions

1. Set `AUTH_METHOD` to your preferred method
2. Configure workspace URLs
3. If using Service Principal or PAT, store credentials in Databricks secrets first
4. Run this cell to configure authentication

In [ ]:
# ============================================================================
# AUTHENTICATION CONFIGURATION
# ============================================================================

# Choose authentication method: "pat", "oauth", "service_principal"
# RECOMMENDED: "oauth" for most use cases
AUTH_METHOD = "oauth"  # Default: OAuth (RECOMMENDED)

# ============================================================================
# WORKSPACE CONFIGURATION
# ============================================================================

# Source Workspace (where dashboards currently exist)
SOURCE_WORKSPACE_URL = "https://your-source-workspace.cloud.databricks.com"

# Target Workspace (where dashboards will be migrated)
TARGET_WORKSPACE_URL = "https://your-target-workspace.cloud.databricks.com"

# ============================================================================
# AUTHENTICATION CREDENTIALS
# ============================================================================

# --- OAuth Authentication (Azure AD) - RECOMMENDED ---
# No configuration needed! Just run: az login
# Or set these environment variables:
#   ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET
# 
# RECOMMENDED for: Interactive use, development, most scenarios
# Advantages: Easy setup, secure, no credential management
# Setup: Run 'az login' in your terminal

# --- Service Principal Authentication ---
# Store credentials in Databricks secrets
# Good for: Production, automated pipelines, CI/CD
# 
# Setup:
# 1. Create service principals in Azure AD
# 2. Store credentials:
#    databricks secrets put --scope migration --key source-sp-client-id
#    databricks secrets put --scope migration --key source-sp-secret
#    databricks secrets put --scope migration --key source-sp-tenant
#    databricks secrets put --scope migration --key target-sp-client-id
#    databricks secrets put --scope migration --key target-sp-secret
#    databricks secrets put --scope migration --key target-sp-tenant

if AUTH_METHOD == "service_principal":
    SOURCE_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="source-sp-client-id")
    SOURCE_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="source-sp-secret")
    SOURCE_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="source-sp-tenant")
    
    TARGET_SP_CLIENT_ID = dbutils.secrets.get(scope="migration", key="target-sp-client-id")
    TARGET_SP_CLIENT_SECRET = dbutils.secrets.get(scope="migration", key="target-sp-secret")
    TARGET_SP_TENANT_ID = dbutils.secrets.get(scope="migration", key="target-sp-tenant")

# --- PAT Token Authentication ---
# Store tokens in Databricks secrets
# Good for: Quick tests, development
# Note: Tokens expire and require rotation
# 
# Setup:
#    databricks secrets put --scope migration --key source-token
#    databricks secrets put --scope migration --key target-token

if AUTH_METHOD == "pat":
    SOURCE_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="source-token")
    TARGET_PAT_TOKEN = dbutils.secrets.get(scope="migration", key="target-token")
    # Alternative for testing ONLY (not recommended):
    # SOURCE_PAT_TOKEN = "dapi..."
    # TARGET_PAT_TOKEN = "dapi..."

# ============================================================================
# DISPLAY CONFIGURATION
# ============================================================================

print("=" * 80)
print("AUTHENTICATION CONFIGURATION")
print("=" * 80)

print(f"\n✅ Authentication Method: {AUTH_METHOD.upper()}")

if AUTH_METHOD == "oauth":
    print("   📋 Using OAuth/Azure AD (RECOMMENDED)")
    print("   ℹ️  Make sure you've run 'az login' or set ARM_* env variables")
    print("   ✨ Advantages: Easy, secure, no credential management")
elif AUTH_METHOD == "service_principal":
    print("   📋 Using Service Principal")
    print("   ℹ️  Good for: Production, automation, CI/CD")
    print("   ✨ Advantages: No token expiration, fine-grained permissions")
elif AUTH_METHOD == "pat":
    print("   📋 Using PAT Tokens")
    print("   ℹ️  Good for: Quick tests")
    print("   ⚠️  Note: Tokens expire and need rotation")

print(f"\n🔗 Workspaces:")
print(f"   Source: {SOURCE_WORKSPACE_URL}")
print(f"   Target: {TARGET_WORKSPACE_URL}")

print("\n" + "=" * 80)
print("✅ Configuration loaded successfully")
print("=" * 80)

## Cell 4: Volume Configuration

**Purpose:** Configure Databricks volume paths for storing migration artifacts.

### Volume Structure

```
/Volumes/[catalog]/[schema]/[volume]/dashboard_migration/
├── mappings/              # CSV mapping files
├── exported/              # Exported .lvdash.json files
├── transformed/           # Transformed .lvdash.json files
└── logs/                  # Migration reports
```

### Instructions

1. Update the volume path below
2. Volume must exist in Unity Catalog
3. Run this cell to configure paths

In [ ]:
# ============================================================================
# VOLUME CONFIGURATION
# ============================================================================

# Base volume path (MUST EXIST - create if needed)
VOLUME_BASE_PATH = "/Volumes/migration_catalog/migration_schema/migration_volume/dashboard_migration"

# Subdirectories (will be created automatically)
MAPPING_CSV_PATH = f"{VOLUME_BASE_PATH}/mappings/catalog_schema_mapping.csv"
EXPORT_PATH = f"{VOLUME_BASE_PATH}/exported"
TRANSFORMED_PATH = f"{VOLUME_BASE_PATH}/transformed"
LOGS_PATH = f"{VOLUME_BASE_PATH}/logs"

# Display configuration
print("=" * 80)
print("VOLUME CONFIGURATION")
print("=" * 80)

print(f"\n📁 Volume Base Path:")
print(f"   {VOLUME_BASE_PATH}")

print(f"\n📂 Subdirectories:")
print(f"   Mappings:    {VOLUME_BASE_PATH}/mappings/")
print(f"   Exported:    {EXPORT_PATH}")
print(f"   Transformed: {TRANSFORMED_PATH}")
print(f"   Logs:        {LOGS_PATH}")

print(f"\n📄 CSV Mapping File:")
print(f"   {MAPPING_CSV_PATH}")

print("\n" + "=" * 80)
print("✅ Volume paths configured")
print("=" * 80)

## Cell 5: Create Volume Structure

**Purpose:** Create the volume directory structure.

**Note:** The volume itself must already exist. This cell only creates subdirectories.

In [ ]:
print("=" * 80)
print("CREATING VOLUME DIRECTORY STRUCTURE")
print("=" * 80)

directories_to_create = [
    f"{VOLUME_BASE_PATH}/mappings",
    EXPORT_PATH,
    TRANSFORMED_PATH,
    LOGS_PATH
]

created = []
existing = []
errors = []

for directory in directories_to_create:
    try:
        # Check if directory already exists
        try:
            dbutils.fs.ls(directory)
            # Directory exists
            existing.append(directory)
            print(f"ℹ️  Already exists: {directory}")
        except Exception:
            # Directory doesn't exist, create it
            dbutils.fs.mkdirs(directory)
            created.append(directory)
            print(f"✅ Created: {directory}")
    except Exception as e:
        errors.append((directory, str(e)))
        print(f"❌ Error with {directory}: {e}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Created: {len(created)}")
print(f"Already existed: {len(existing)}")
print(f"Errors: {len(errors)}")

if errors:
    print("\n⚠️  ERRORS DETECTED:")
    for dir_path, error in errors:
        print(f"   {dir_path}: {error}")
    print("\n❌ Please fix errors before proceeding")
else:
    print("\n✅ Volume structure ready")

## Cell 6: Create CSV Mapping Template

**Purpose:** Create a template CSV file for catalog/schema/table mappings.

### CSV Format

| Column | Description | Example |
|--------|-------------|----------|
| old_catalog | Source catalog name | dev_catalog |
| old_schema | Source schema name | bronze_layer |
| old_table | Source table name | customers |
| new_catalog | Target catalog name | prod_catalog |
| new_schema | Target schema name | gold_layer |
| new_table | Target table name | customers |
| old_volume | Source volume name (optional) | dev_files |
| new_volume | Target volume name (optional) | prod_files |

In [ ]:
# CSV template content
csv_template = """old_catalog,old_schema,old_table,new_catalog,new_schema,new_table,old_volume,new_volume
dev_catalog,bronze_layer,customers,prod_catalog,gold_layer,customers,dev_files,prod_files
dev_catalog,bronze_layer,orders,prod_catalog,gold_layer,orders,dev_files,prod_files
dev_catalog,bronze_layer,products,prod_catalog,gold_layer,products,dev_files,prod_files
staging_cat,raw_data,events,prod_catalog,curated_data,events,staging_files,prod_files
staging_cat,raw_data,clicks,prod_catalog,analytics,clicks,staging_files,analytics_files
test_catalog,sandbox,transactions,prod_catalog,finance,transactions,test_vol,prod_vol
"""

# Write template to volume
try:
    # Convert volume path to local path
    # Write using dbutils.fs (serverless-compatible)
    dbutils.fs.put(MAPPING_CSV_PATH, csv_template, overwrite=True)
    
    print("=" * 80)
    print("CSV MAPPING TEMPLATE CREATED")
    print("=" * 80)
    print(f"\n✅ Template created at: {MAPPING_CSV_PATH}")
    
    # Display template content
    print("\n📋 Template content (example mappings):")
    print("\n" + csv_template)
    
    print("=" * 80)
    print("NEXT STEPS")
    print("=" * 80)
    print("1. Edit the CSV file with your actual catalog/schema/table mappings")
    print("2. You can edit it directly in the volume or download/upload")
    print("3. Each row maps one source table to a target table")
    print("4. Leave table columns empty for schema-level mappings")
    print("5. old_volume/new_volume are optional for volume path replacements")
    
except Exception as e:
    print(f"❌ Error creating template: {e}")
    print("\nYou can create the CSV file manually at:")
    print(f"   {MAPPING_CSV_PATH}")

## Cell 7: Test Workspace Connectivity

**Purpose:** Validate workspace client creation (lightweight test for serverless).

**Note:** Full API connectivity tested during export in Notebook 2.

In [ ]:
def create_workspace_client(workspace_url: str, is_source: bool = True) -> WorkspaceClient:
    """
    Create WorkspaceClient with configured authentication method.
    
    Args:
        workspace_url: Databricks workspace URL
        is_source: True for source workspace, False for target
        
    Returns:
        Configured WorkspaceClient
    """
    if AUTH_METHOD == "service_principal":
        if is_source:
            return WorkspaceClient(
                host=workspace_url,
                client_id=SOURCE_SP_CLIENT_ID,
                client_secret=SOURCE_SP_CLIENT_SECRET,
                azure_tenant_id=SOURCE_SP_TENANT_ID
            )
        else:
            return WorkspaceClient(
                host=workspace_url,
                client_id=TARGET_SP_CLIENT_ID,
                client_secret=TARGET_SP_CLIENT_SECRET,
                azure_tenant_id=TARGET_SP_TENANT_ID
            )
    
    elif AUTH_METHOD == "oauth":
        # OAuth uses environment variables or Azure CLI
        return WorkspaceClient(host=workspace_url)
    
    elif AUTH_METHOD == "pat":
        token = SOURCE_PAT_TOKEN if is_source else TARGET_PAT_TOKEN
        return WorkspaceClient(host=workspace_url, token=token)
    
    else:
        raise ValueError(f"Unknown auth method: {AUTH_METHOD}")

# Test connectivity
print("=" * 80)
print("TESTING WORKSPACE CONNECTIVITY")
print("=" * 80)

results = {"source": False, "target": False}
errors = {}

# Test source workspace
print("\n🔍 Testing source workspace...")
try:
    source_client = create_workspace_client(SOURCE_WORKSPACE_URL, is_source=True)
    # Client created successfully (skip API call on serverless)
    print(f"   ✅ Source client created")
    print(f"   Host: {SOURCE_WORKSPACE_URL}")
    results["source"] = True
except Exception as e:
    print(f"   ❌ Failed to connect to source: {e}")
    errors["source"] = str(e)

# Test target workspace
print("\n🔍 Testing target workspace...")
try:
    target_client = create_workspace_client(TARGET_WORKSPACE_URL, is_source=False)
    # Client created successfully (skip API call on serverless)
    print(f"   ✅ Target client created")
    print(f"   Host: {TARGET_WORKSPACE_URL}")
    results["target"] = True
except Exception as e:
    print(f"   ❌ Failed to connect to target: {e}")
    errors["target"] = str(e)

# Summary
print("\n" + "=" * 80)
print("CONNECTIVITY TEST SUMMARY")
print("=" * 80)

if all(results.values()):
    print("\n✅ SUCCESS: Both workspace clients created successfully")
    print("\n💡 Full API connectivity will be tested during export (Notebook 2)")
    print("\n👉 You can proceed to Notebook 2: Export & Transform")
else:
    print("\n❌ FAILED: One or more workspaces are not accessible")
    print("\n⚠️  Please fix the following issues:")
    for workspace, error in errors.items():
        print(f"\n{workspace.upper()}:")
        print(f"   {error}")
    
    if AUTH_METHOD == "oauth":
        print("\n💡 OAuth troubleshooting:")
        print("   - Run 'az login' in your terminal")
        print("   - Or set ARM_CLIENT_ID, ARM_TENANT_ID, ARM_CLIENT_SECRET env variables")
    elif AUTH_METHOD == "service_principal":
        print("\n💡 Service Principal troubleshooting:")
        print("   - Verify credentials are stored in Databricks secrets")
        print("   - Check service principal has workspace access")
    elif AUTH_METHOD == "pat":
        print("\n💡 PAT troubleshooting:")
        print("   - Verify tokens are stored in Databricks secrets")
        print("   - Check tokens haven't expired")
        print("   - Verify tokens have workspace access")

## Cell 8: Verify Volume and CSV Mapping

**Purpose:** Verify that the volume structure and CSV mapping file are ready.

In [ ]:
print("=" * 80)
print("VERIFICATION CHECKLIST")
print("=" * 80)

checks = []

# Check 1: Volume base path exists
print("\n1️⃣  Checking volume base path...")
try:
    dbutils.fs.ls(VOLUME_BASE_PATH)
    print(f"   ✅ Volume base path exists: {VOLUME_BASE_PATH}")
    checks.append(True)
except Exception as e:
    print(f"   ❌ Volume base path not accessible: {e}")
    checks.append(False)

# Check 2: Subdirectories exist
print("\n2️⃣  Checking subdirectories...")
subdirs = ["mappings", "exported", "transformed", "logs"]
subdir_checks = []
for subdir in subdirs:
    try:
        dbutils.fs.ls(f"{VOLUME_BASE_PATH}/{subdir}")
        print(f"   ✅ {subdir}/")
        subdir_checks.append(True)
    except:
        print(f"   ❌ {subdir}/ - not found")
        subdir_checks.append(False)
checks.append(all(subdir_checks))

# Check 3: CSV mapping file exists
print("\n3️⃣  Checking CSV mapping file...")
try:
    # Read using dbutils.fs (serverless-compatible)
    content = dbutils.fs.head(MAPPING_CSV_PATH, 10485760)
    
    # Parse CSV to verify structure
    lines = content.strip().split('\n')
    reader = csv.DictReader(lines)
    mappings = list(reader)
    
    print(f"   ✅ CSV file exists: {MAPPING_CSV_PATH}")
    print(f"   📊 Found {len(mappings)} mapping row(s)")
    
    # Display first few mappings
    if mappings:
        print("\n   Sample mappings:")
        for i, mapping in enumerate(mappings[:3], 1):
            print(f"      {i}. {mapping['old_catalog']}.{mapping['old_schema']}.{mapping['old_table']} → "
                  f"{mapping['new_catalog']}.{mapping['new_schema']}.{mapping['new_table']}")
        if len(mappings) > 3:
            print(f"      ... and {len(mappings) - 3} more")
    
    checks.append(True)
except Exception as e:
    print(f"   ❌ CSV file not found or invalid: {e}")
    checks.append(False)

# Check 4: Workspace connectivity
print("\n4️⃣  Checking workspace connectivity...")
if results.get("source") and results.get("target"):
    print("   ✅ Both workspaces are accessible")
    checks.append(True)
else:
    print("   ❌ Workspace connectivity issues detected")
    checks.append(False)

# Final summary
print("\n" + "=" * 80)
print("SETUP STATUS")
print("=" * 80)

if all(checks):
    print("\n✅ ALL CHECKS PASSED!")
    print("\n🎉 Setup complete! You're ready for the next step.")
    print("\n" + "=" * 80)
    print("NEXT STEP")
    print("=" * 80)
    print("\n👉 Open Notebook 2: 02_Export_and_Transform.ipynb")
    print("\n   This notebook will:")
    print("   - Export dashboards from source workspace")
    print("   - Apply CSV mappings to transform references")
    print("   - Prepare dashboards for import")
else:
    print("\n⚠️  SETUP INCOMPLETE")
    print("\nPlease fix the failed checks above before proceeding.")
    print("\nFailed checks:")
    check_names = ["Volume base path", "Subdirectories", "CSV mapping file", "Workspace connectivity"]
    for i, (check_name, passed) in enumerate(zip(check_names, checks), 1):
        if not passed:
            print(f"   {i}. {check_name}")

## Setup Complete! 🎉

### What You've Accomplished

- ✅ Installed required libraries
- ✅ Configured authentication
- ✅ Created volume structure
- ✅ Created CSV mapping template
- ✅ Tested workspace connectivity
- ✅ Verified all prerequisites

### Next Steps

1. **Edit your CSV mapping file** with actual catalog/schema/table mappings
2. **Open Notebook 2**: `02_Export_and_Transform.ipynb`
3. Continue with dashboard export and transformation

### Configuration Summary

Your configuration has been saved in this notebook. Key settings:
- Authentication: Configured
- Volume: Ready
- CSV Mappings: Template created
- Workspaces: Connected

---

**Questions or Issues?** Refer to the comprehensive guide document.